# 00 Setup And Config
Notebook-Interface Einstieg: Stage, Pfade, Environment, Systemcheck, `run_id`.

In [1]:
from pathlib import Path
import os
import json
import uuid
from datetime import datetime
import platform
import shutil

import pandas as pd
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
SEED_OVERRIDE = None
DEVICE = "auto"
USE_STUB_EMBEDDINGS = False  # nur für lokale Smoke-Checks ohne torch/chars2vec
PREFER_PRECOMPUTED_ADS = True

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "configs").exists():
            return candidate.resolve()
    return start.resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation


In [2]:
# Optional .env laden
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
    print(".env loaded")
except Exception:
    print("python-dotenv nicht installiert - überspringe .env loading")

hf_token = os.getenv("HF_TOKEN", "")
print("HF_TOKEN gesetzt:", bool(hf_token))


.env loaded
HF_TOKEN gesetzt: True


In [3]:
from src.common.config import load_yaml, resolve_paths_config

paths_cfg = resolve_paths_config(load_yaml(PROJECT_ROOT / "configs/paths.colab.yaml"), project_root=PROJECT_ROOT)
model_cfg = load_yaml(PROJECT_ROOT / "configs/model/nand_best.yaml")
run_cfg = load_yaml(PROJECT_ROOT / f"configs/runs/{RUN_STAGE}.yaml")
if SEED_OVERRIDE is not None:
    run_cfg["seed"] = int(SEED_OVERRIDE)

config_preview = {
    "run_stage": RUN_STAGE,
    "run_seed": run_cfg.get("seed"),
    "subset_target_mentions": run_cfg.get("subset_target_mentions"),
    "raw_lspo": paths_cfg["data"]["raw_lspo_parquet"],
    "raw_ads_publications": paths_cfg["data"]["raw_ads_publications"],
    "raw_ads_references": paths_cfg["data"]["raw_ads_references"],
    "model": model_cfg.get("name"),
    "loss": model_cfg.get("training", {}).get("loss"),
    "arch": f"{model_cfg['training']['input_dim']}->{model_cfg['training']['hidden_dim']}->{model_cfg['training']['output_dim']}",
}

pd.DataFrame([config_preview]).T.rename(columns={0: "value"})


,value
run_stage,smoke
run_seed,11
subset_target_mentions,1000
raw_lspo,data/raw/lspo/LSPO_v1.parquet
raw_ads_publications,data/raw/ads/queried_publications_lang_transla...
raw_ads_references,data/raw/ads/queried_references_lang_translate...
model,nand_best
loss,infonce
arch,818->1024->256


In [4]:
# Systemcheck (RAM/GPU/Storage)
try:
    import psutil
    vm = psutil.virtual_memory()
    ram_gb = vm.total / (1024**3)
except Exception:
    ram_gb = None

free_gb = shutil.disk_usage(PROJECT_ROOT).free / (1024**3)

gpu_name = None
cuda_ok = False
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
except Exception:
    pass

print("RAM_GB:", ram_gb)
print("FREE_DISK_GB:", round(free_gb, 2))
print("CUDA_AVAILABLE:", cuda_ok)
print("GPU:", gpu_name)
print("Python:", platform.python_version())


RAM_GB: 43.82659149169922
FREE_DISK_GB: 49.11
CUDA_AVAILABLE: False
GPU: None
Python: 3.10.13


In [5]:
# run_id + artefaktordner
RUN_ID = f"{RUN_STAGE}_{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"

run_dirs = {
    "metrics": PROJECT_ROOT / "artifacts/metrics" / RUN_ID,
    "checkpoints": PROJECT_ROOT / "artifacts/checkpoints" / RUN_ID,
    "pair_scores": PROJECT_ROOT / "artifacts/pair_scores" / RUN_ID,
    "clusters": PROJECT_ROOT / "artifacts/clusters" / RUN_ID,
    "embeddings": PROJECT_ROOT / "artifacts/embeddings" / RUN_ID,
    "subset_cache": PROJECT_ROOT / "data/subsets/cache" / RUN_ID,
}
for p in run_dirs.values():
    p.mkdir(parents=True, exist_ok=True)

context = {
    "run_id": RUN_ID,
    "run_stage": RUN_STAGE,
    "device": DEVICE,
    "use_stub_embeddings": USE_STUB_EMBEDDINGS,
    "prefer_precomputed_ads": PREFER_PRECOMPUTED_ADS,
}
with (run_dirs["metrics"] / "00_context.json").open("w", encoding="utf-8") as f:
    json.dump(context, f, indent=2)

print("RUN_ID:", RUN_ID)
print(json.dumps({k: str(v) for k, v in run_dirs.items()}, indent=2))


RUN_ID: smoke_20260212T164150Z_e596ad31
{
  "metrics": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\metrics\\smoke_20260212T164150Z_e596ad31",
  "checkpoints": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\checkpoints\\smoke_20260212T164150Z_e596ad31",
  "pair_scores": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\pair_scores\\smoke_20260212T164150Z_e596ad31",
  "clusters": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\clusters\\smoke_20260212T164150Z_e596ad31",
  "embeddings": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\embeddings\\smoke_20260212T164150Z_e596ad31",
  "subset_cache": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsst